# 🧪 MedNova AI — Tests S3 + S4 + S5


In [2]:
# ── CELL 1 : Setup
import os, json, joblib
import numpy as np
import pandas as pd

MODELS_DIR = os.path.abspath('../models')
JSON_DIR   = os.path.abspath('../outputs/json')
BI_DIR     = os.path.abspath('../outputs/bi')
CLUST_DIR  = os.path.abspath('../outputs/clustering')
SHAP_DIR   = os.path.abspath('../outputs/shap_plots')

passed = []
failed = []

def run_test(name, fn):
    try:
        fn()
        passed.append(name)
        print(f'  ✅ {name}')
    except Exception as e:
        failed.append((name, str(e)))
        print(f'  ❌ {name}')
        print(f'     → {e}')

print('✅ Setup OK')
print(f'   MODELS_DIR : {MODELS_DIR}')
print(f'   JSON_DIR   : {JSON_DIR}')

✅ Setup OK
   MODELS_DIR : C:\Users\HP\Desktop\ing4\MedNova\MedNova\models
   JSON_DIR   : C:\Users\HP\Desktop\ing4\MedNova\MedNova\outputs\json


In [3]:
# ── CELL 2 : Tests Semaine 3
print('\n' + '='*50)
print('  🔬 SEMAINE 3 — Complications + SHAP')
print('='*50)

def t_s3_models_exist():
    for f in ['complication_model.pkl','shap_explainer.pkl','scaler.pkl',
              'num_imputer.pkl','feature_columns.pkl','model_metadata.pkl']:
        assert os.path.exists(os.path.join(MODELS_DIR, f)), f'Manquant: {f}'

def t_s3_model_loadable():
    model = joblib.load(os.path.join(MODELS_DIR, 'complication_model.pkl'))
    assert hasattr(model, 'predict_proba')

def t_s3_metadata():
    meta = joblib.load(os.path.join(MODELS_DIR, 'model_metadata.pkl'))
    assert meta['auc_roc'] > 0.7, f'AUC trop faible: {meta["auc_roc"]}'
    assert meta['semaine'] == 3
    print(f'     AUC={meta["auc_roc"]} | Modèle={meta["model_name"]}')

def t_s3_predict():
    model   = joblib.load(os.path.join(MODELS_DIR, 'complication_model.pkl'))
    scaler  = joblib.load(os.path.join(MODELS_DIR, 'scaler.pkl'))
    imputer = joblib.load(os.path.join(MODELS_DIR, 'num_imputer.pkl'))
    cols    = joblib.load(os.path.join(MODELS_DIR, 'feature_columns.pkl'))
    patient = pd.DataFrame([{c: 0.0 for c in cols}])
    scaled  = pd.DataFrame(scaler.transform(imputer.transform(patient)), columns=cols)
    prob    = float(model.predict_proba(scaled)[0][1])
    assert 0.0 <= prob <= 1.0
    print(f'     prob={prob:.3f}')

def t_s3_predict_10():
    model   = joblib.load(os.path.join(MODELS_DIR, 'complication_model.pkl'))
    scaler  = joblib.load(os.path.join(MODELS_DIR, 'scaler.pkl'))
    imputer = joblib.load(os.path.join(MODELS_DIR, 'num_imputer.pkl'))
    cols    = joblib.load(os.path.join(MODELS_DIR, 'feature_columns.pkl'))
    np.random.seed(42)
    X       = pd.DataFrame(np.random.randn(10, len(cols)), columns=cols)
    scaled  = pd.DataFrame(scaler.transform(imputer.transform(X)), columns=cols)
    probs   = model.predict_proba(scaled)[:, 1]
    assert len(probs) == 10
    assert all(0 <= p <= 1 for p in probs)
    print(f'     probs min={probs.min():.3f} max={probs.max():.3f}')

def t_s3_risk_levels():
    def r(p):
        if p < 0.25: return 'Faible'
        elif p < 0.50: return 'Modere'
        elif p < 0.75: return 'Eleve'
        else: return 'Critique'
    assert r(0.10)=='Faible' and r(0.35)=='Modere'
    assert r(0.60)=='Eleve'  and r(0.85)=='Critique'

run_test('S3 — Models exist',    t_s3_models_exist)
run_test('S3 — Model loadable',  t_s3_model_loadable)
run_test('S3 — Metadata AUC',    t_s3_metadata)
run_test('S3 — Predict patient', t_s3_predict)
run_test('S3 — Predict 10 pts',  t_s3_predict_10)
run_test('S3 — Risk levels',     t_s3_risk_levels)


  🔬 SEMAINE 3 — Complications + SHAP
  ✅ S3 — Models exist
  ✅ S3 — Model loadable
     AUC=0.90897 | Modèle=GradientBoosting
  ✅ S3 — Metadata AUC
  ❌ S3 — Predict patient
     → The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- ca
- dnr
- dzclass
- dzgroup
- income
- ...

  ❌ S3 — Predict 10 pts
     → The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- ca
- dnr
- dzclass
- dzgroup
- income
- ...

  ✅ S3 — Risk levels


In [4]:
# ── CELL 3 : Tests Semaine 4
print('\n' + '='*50)
print('  🔵 SEMAINE 4 — Clustering + KNN')
print('='*50)

def t_s4_models_exist():
    for f in ['kmeans.pkl','pca_2d.pkl','nearest_neighbors.pkl',
              'clust_scaler.pkl','clust_imputer.pkl','cluster_profiles.pkl']:
        assert os.path.exists(os.path.join(MODELS_DIR, f)), f'Manquant: {f}'

def t_s4_kmeans():
    kmeans = joblib.load(os.path.join(MODELS_DIR, 'kmeans.pkl'))
    scaler = joblib.load(os.path.join(MODELS_DIR, 'clust_scaler.pkl'))
    n      = scaler.mean_.shape[0]
    c      = int(kmeans.predict(scaler.transform(np.zeros((1,n))))[0])
    assert 0 <= c < kmeans.n_clusters
    print(f'     k={kmeans.n_clusters} | cluster test={c}')

def t_s4_pca():
    pca    = joblib.load(os.path.join(MODELS_DIR, 'pca_2d.pkl'))
    scaler = joblib.load(os.path.join(MODELS_DIR, 'clust_scaler.pkl'))
    n      = scaler.mean_.shape[0]
    result = pca.transform(scaler.transform(np.random.randn(5, n)))
    assert result.shape == (5, 2)
    print(f'     PCA shape: {result.shape} ✓')

def t_s4_knn():
    knn    = joblib.load(os.path.join(MODELS_DIR, 'nearest_neighbors.pkl'))
    scaler = joblib.load(os.path.join(MODELS_DIR, 'clust_scaler.pkl'))
    n      = scaler.mean_.shape[0]
    dist, idx = knn.kneighbors(scaler.transform(np.zeros((1,n))))
    assert dist.shape[1] == knn.n_neighbors
    print(f'     KNN n_neighbors={knn.n_neighbors} | dist min={dist.min():.3f}')

def t_s4_profiles():
    profiles = joblib.load(os.path.join(MODELS_DIR, 'cluster_profiles.pkl'))
    kmeans   = joblib.load(os.path.join(MODELS_DIR, 'kmeans.pkl'))
    assert len(profiles) == kmeans.n_clusters
    for p in profiles:
        assert 0 <= p['complication_rate'] <= 100
        assert p['n_patients'] > 0
    print(f'     {len(profiles)} profils valides')

def t_s4_json():
    for f in ['clusters_overview.json','pca_scatter.json',
              'cluster_distribution.json','similar_patients_example.json']:
        path = os.path.join(JSON_DIR, f)
        assert os.path.exists(path), f'Manquant: {f}'
        with open(path) as fp:
            json.load(fp)

def t_s4_scatter():
    with open(os.path.join(JSON_DIR, 'pca_scatter.json')) as f:
        data = json.load(f)
    assert 'points' in data and len(data['points']) > 0
    pt = data['points'][0]
    assert 'x' in pt and 'y' in pt and 'cluster' in pt
    print(f'     {len(data["points"])} points dans scatter')

run_test('S4 — Models exist',    t_s4_models_exist)
run_test('S4 — KMeans predict',  t_s4_kmeans)
run_test('S4 — PCA 2D shape',    t_s4_pca)
run_test('S4 — KNN neighbors',   t_s4_knn)
run_test('S4 — Cluster profiles',t_s4_profiles)
run_test('S4 — JSON files',      t_s4_json)
run_test('S4 — Scatter JSON',    t_s4_scatter)


  🔵 SEMAINE 4 — Clustering + KNN
  ✅ S4 — Models exist


C:\Users\HP\Desktop\ing4\MedNova\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


     k=5 | cluster test=3
  ✅ S4 — KMeans predict
     PCA shape: (5, 2) ✓
  ✅ S4 — PCA 2D shape
     KNN n_neighbors=11 | dist min=105.093
  ✅ S4 — KNN neighbors
     5 profils valides
  ✅ S4 — Cluster profiles
  ✅ S4 — JSON files
     2000 points dans scatter
  ✅ S4 — Scatter JSON


C:\Users\HP\Desktop\ing4\MedNova\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\HP\Desktop\ing4\MedNova\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [5]:
# ── CELL 4 : Tests Semaine 5
print('\n' + '='*50)
print('  📊 SEMAINE 5 — KPIs + Forecasting')
print('='*50)

def t_s5_json_exist():
    for f in ['kpis_global.json','kpis_pathologie.json','kpis_service.json',
              'kpis_age.json','kpis_monthly.json','forecasts.json','dashboard_summary.json']:
        assert os.path.exists(os.path.join(JSON_DIR, f)), f'Manquant: {f}'

def t_s5_kpis_values():
    with open(os.path.join(JSON_DIR, 'kpis_global.json')) as f:
        data = json.load(f)
    g = data['global']
    assert 0 < g['taux_succes_pct']       < 100
    assert 0 < g['taux_complication_pct'] < 100
    assert g['duree_sejour_moy_j'] > 0
    assert g['n_patients'] == 9105
    print(f'     succes={g["taux_succes_pct"]}% | compl={g["taux_complication_pct"]}%')
    print(f'     duree={g["duree_sejour_moy_j"]}j | cout={g["cout_moyen_eur"]}€')

def t_s5_alerts():
    with open(os.path.join(JSON_DIR, 'kpis_global.json')) as f:
        data = json.load(f)
    alerts = data['alerts']
    assert len(alerts) > 0
    critiques = sum(1 for a in alerts if 'CRITIQUE' in a['niveau'])
    warnings  = sum(1 for a in alerts if 'WARNING'  in a['niveau'])
    print(f'     {critiques} critiques | {warnings} warnings')

def t_s5_pathologie():
    with open(os.path.join(JSON_DIR, 'kpis_pathologie.json')) as f:
        data = json.load(f)
    paths = data['pathologies']
    assert len(paths) >= 5
    print(f'     {len(paths)} pathologies | top: {paths[0]["pathologie"]} ({paths[0]["taux_complication_pct"]}%)')

def t_s5_age_trend():
    with open(os.path.join(JSON_DIR, 'kpis_age.json')) as f:
        data = json.load(f)
    ages   = data['age_groups']
    compls = [a['taux_complication_pct'] for a in ages]
    assert compls[-1] > compls[0]
    print(f'     <40: {compls[0]}% → 75+: {compls[-1]}% (tendance croissante ✓)')

def t_s5_monthly():
    with open(os.path.join(JSON_DIR, 'kpis_monthly.json')) as f:
        data = json.load(f)
    monthly = data['monthly']
    assert len(monthly) >= 12
    print(f'     {len(monthly)} mois historique')

def t_s5_forecasts():
    with open(os.path.join(JSON_DIR, 'forecasts.json')) as f:
        data = json.load(f)
    fc = data['forecasts']
    assert len(fc) >= 1
    for kpi, forecast in fc.items():
        assert forecast['mae'] < 50
        for lo, hi in zip(forecast['conf_int_low'], forecast['conf_int_high']):
            assert lo <= hi
        print(f'     {kpi}: MAE={forecast["mae"]} | +{forecast["n_forecast"]}m={forecast["previsions"][-1]}')

def t_s5_dashboard():
    with open(os.path.join(JSON_DIR, 'dashboard_summary.json')) as f:
        data = json.load(f)
    assert 'kpis_snapshot' in data
    assert 'services'      in data
    print(f'     {data["n_alertes_critiques"]} critiques | {data["n_alertes_warning"]} warnings')

run_test('S5 — JSON files exist',  t_s5_json_exist)
run_test('S5 — KPIs values',       t_s5_kpis_values)
run_test('S5 — Alerts',            t_s5_alerts)
run_test('S5 — Pathologie',        t_s5_pathologie)
run_test('S5 — Age trend',         t_s5_age_trend)
run_test('S5 — Monthly series',    t_s5_monthly)
run_test('S5 — Forecasts ARIMA',   t_s5_forecasts)
run_test('S5 — Dashboard summary', t_s5_dashboard)


  📊 SEMAINE 5 — KPIs + Forecasting
  ✅ S5 — JSON files exist
     succes=72.95% | compl=43.25%
     duree=17.7j | cout=22781.0€
  ✅ S5 — KPIs values
     1 critiques | 4 warnings
  ✅ S5 — Alerts
     8 pathologies | top: Coma (73.66%)
  ✅ S5 — Pathologie
     <40: 32.15% → 75+: 49.25% (tendance croissante ✓)
  ✅ S5 — Age trend
     56 mois historique
  ✅ S5 — Monthly series
     taux_complication: MAE=2.8008 | +6m=43.34
     taux_succes: MAE=3.1482 | +6m=72.93
     duree_sejour_moy: MAE=1.0495 | +3m=17.695
  ✅ S5 — Forecasts ARIMA
     1 critiques | 4 warnings
  ✅ S5 — Dashboard summary


In [6]:
# ── CELL 5 : Tests Visualisations PNG
print('\n' + '='*50)
print('  🖼️  VISUALISATIONS — PNG')
print('='*50)

def t_png_s3():
    for f in ['03_shap_summary.png','04_shap_importance_bar.png']:
        assert os.path.exists(os.path.join(SHAP_DIR, f)), f'PNG manquant: {f}'
        size = os.path.getsize(os.path.join(SHAP_DIR, f))
        print(f'     {f} ({size//1024} KB)')

def t_png_s4():
    for f in ['02_clusters_pca2d.png','03_cluster_profiles_heatmap.png']:
        assert os.path.exists(os.path.join(CLUST_DIR, f)), f'PNG manquant: {f}'
        size = os.path.getsize(os.path.join(CLUST_DIR, f))
        print(f'     {f} ({size//1024} KB)')

def t_png_s5():
    for f in ['00_kpis_globaux.png','01_kpis_pathologie.png',
              '03_evolution_mensuelle.png','05_alertes.png']:
        assert os.path.exists(os.path.join(BI_DIR, f)), f'PNG manquant: {f}'
        size = os.path.getsize(os.path.join(BI_DIR, f))
        print(f'     {f} ({size//1024} KB)')

run_test('PNG S3 — SHAP plots',    t_png_s3)
run_test('PNG S4 — Clustering',    t_png_s4)
run_test('PNG S5 — BI dashboard',  t_png_s5)


  🖼️  VISUALISATIONS — PNG
     03_shap_summary.png (209 KB)
     04_shap_importance_bar.png (67 KB)
  ✅ PNG S3 — SHAP plots
     02_clusters_pca2d.png (431 KB)
     03_cluster_profiles_heatmap.png (81 KB)
  ✅ PNG S4 — Clustering
     00_kpis_globaux.png (71 KB)
     01_kpis_pathologie.png (83 KB)
     03_evolution_mensuelle.png (187 KB)
     05_alertes.png (62 KB)
  ✅ PNG S5 — BI dashboard


In [7]:
# ── CELL 6 : Rapport final
total = len(passed) + len(failed)

print('\n' + '='*50)
print('  📋 RAPPORT FINAL')
print('='*50)
print(f'  ✅ Passés  : {len(passed)} / {total}')
print(f'  ❌ Échoués : {len(failed)} / {total}')

if failed:
    print('\n  Tests échoués :')
    for name, err in failed:
        print(f'  ❌ {name}')
        print(f'     → {err}')
else:
    print('\n  🎉 Tous les tests passés!')
    print('  Le projet S3 + S4 + S5 est validé ✅')
print('='*50)


  📋 RAPPORT FINAL
  ✅ Passés  : 22 / 24
  ❌ Échoués : 2 / 24

  Tests échoués :
  ❌ S3 — Predict patient
     → The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- ca
- dnr
- dzclass
- dzgroup
- income
- ...

  ❌ S3 — Predict 10 pts
     → The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- ca
- dnr
- dzclass
- dzgroup
- income
- ...

